In [1]:
import os
import sys
import numpy as np
import json
import random
import collections


import torch
import torch.optim as optim
import torchvision.utils as vutils

import torch.nn as nn
import torch.nn.functional as F



## パスの設定

### cifar100

In [2]:
# ベース部分のパス
ckpt_path = "/home/kouyou/ContinualLearning/repexp/NeurIPS2024-PRL/checkpoint"

# cifar100のbaseline用パス
base_cifar100_path = "baseline/cifar100"

# baseline
method = "baseline"
baseline_path = os.path.join(ckpt_path, method, base_cifar100_path, "50/5/10_10_0_0_0")

# baseline_mu
method = "baseline_mu"
baseline_mu_path = os.path.join(ckpt_path, method, base_cifar100_path, "50/5/10_10_0_0_1.0")
# print("baseline_mu_path: ", baseline_mu_path)

# prl2
method = "prl2"
prl_path = os.path.join(ckpt_path, method, base_cifar100_path, "50/5/10_10_0.1_2_1.0")
# print("prl_path: ", prl_path)

# prl_mu
method = "prl-mu"
prl_mu_path = os.path.join(ckpt_path, method, base_cifar100_path, "50/5/10_10_0.1_2_1.0")
# print("prl_mu_path: ", prl_mu_path)

### tiny-imagenet

In [5]:
# ベース部分のパス
ckpt_path = "/home/kouyou/ContinualLearning/repexp/NeurIPS2024-PRL/checkpoint"

# tiny-imagenetのbaseline用パス
base_cifar100_path = "baseline/tiny200"

# baseline
method = "baseline"
baseline_path = os.path.join(ckpt_path, method, base_cifar100_path, "100/10/15_15_0.1_1_1")

## 色々と設定

In [6]:
# プロジェクト root を sys.path に追加
project_root = "/home/kouyou/ContinualLearning/repexp/NeurIPS2024-PRL"
sys.path.append(project_root)

from utils import factory
import models

# 学習時のクラス順序（この順序で0~99のラベルを新しく割り当てている）
class_order = [68, 56, 78, 8, 23, 84, 90, 65, 74, 76,
               40, 89, 3, 92, 55, 9, 26, 80, 43, 38,
               58, 70, 77, 1, 85, 19, 17, 50, 28, 53,
               13, 81, 45, 82, 6, 59, 83, 16, 15, 44,
               91, 41, 72, 60, 79, 52, 20, 10, 31, 54,
               37, 95, 14, 71, 96, 98, 97, 2, 64, 66,
               42, 22, 35, 86, 24, 34, 87, 21, 99, 0,
               88, 27, 18, 94, 11, 12, 47, 25, 30, 46,
               62, 69, 36, 61, 7, 63, 75, 5, 32, 4,
               51, 48, 73, 93, 39, 67, 29, 49, 57, 33]

# --- 1) PRL_MU の設定を読む ---
with open(os.path.join(project_root, "exps", "BASELINE", "cifar.json")) as f:
    args = json.load(f)

args["device"] = ["0"]          # 必要に応じて
args["model_name"] = "baseline"   # 念のため明示


# --- 2) learner とネットワークの作成 ---
learner = factory.get_model(args["model_name"], args)
net = learner._network


# --- 3) checkpoint の読み込み ---
ckpt_dir = baseline_path                           # さっきのノートで定義したものを使う
ckpt_file = os.path.join(ckpt_dir, "phase0.pkl")  # 実際のファイル名に合わせて変更

ckpt = torch.load(ckpt_file, map_location="cuda:0")
state_dict = ckpt["model_state_dict"]
print(state_dict.keys())

# ここがポイント：fc の出力次元を checkpoint から取得
num_outputs = state_dict["fc.weight"].shape[0]
print("num_outputs: ", num_outputs)

# 先に fc をその次元で作っておく
net.update_fc(num_outputs)

# そのあとに state_dict を読み込む
net.load_state_dict(state_dict)

net.cuda().eval()

# 忘却クラスや class_order をノート側で使うなら:
forget_classes = ckpt.get("forget_classes", None)
class_order = ckpt.get("_class_order", None)
print(class_order)

odict_keys(['convnet.conv1.0.weight', 'convnet.conv1.1.weight', 'convnet.conv1.1.bias', 'convnet.conv1.1.running_mean', 'convnet.conv1.1.running_var', 'convnet.conv1.1.num_batches_tracked', 'convnet.layer1.0.conv1.weight', 'convnet.layer1.0.bn1.weight', 'convnet.layer1.0.bn1.bias', 'convnet.layer1.0.bn1.running_mean', 'convnet.layer1.0.bn1.running_var', 'convnet.layer1.0.bn1.num_batches_tracked', 'convnet.layer1.0.conv2.weight', 'convnet.layer1.0.bn2.weight', 'convnet.layer1.0.bn2.bias', 'convnet.layer1.0.bn2.running_mean', 'convnet.layer1.0.bn2.running_var', 'convnet.layer1.0.bn2.num_batches_tracked', 'convnet.layer1.1.conv1.weight', 'convnet.layer1.1.bn1.weight', 'convnet.layer1.1.bn1.bias', 'convnet.layer1.1.bn1.running_mean', 'convnet.layer1.1.bn1.running_var', 'convnet.layer1.1.bn1.num_batches_tracked', 'convnet.layer1.1.conv2.weight', 'convnet.layer1.1.bn2.weight', 'convnet.layer1.1.bn2.bias', 'convnet.layer1.1.bn2.running_mean', 'convnet.layer1.1.bn2.running_var', 'convnet.layer

<ipython-input-6-7f0409de76be>:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_file, map_location="cuda:0")


In [7]:
input = torch.randn([1, 3, 32, 32]).cuda()
output = net(input)

In [8]:
# print(output["logits"].shape)
# print(output["logits"])

## Hookの設定

In [9]:
class DeepInversionFeatureHook():
    '''
    Implementation of the forward hook to track feature statistics and compute a loss on them.
    Will compute mean and variance, and will use l2 as a loss
    '''

    def __init__(self, module):
        self.hook = module.register_forward_hook(self.hook_fn)


    def hook_fn(self, module, input, output):
        # hook co compute deepinversion's feature distribution regularization
        nch = input[0].shape[1]

        mean = input[0].mean([0, 2, 3])
        var = input[0].permute(1, 0, 2, 3).contiguous().view([nch, -1]).var(1, unbiased=False)

        # forcing mean and variance to match between two distributions
        # other ways might work better, e.g. KL divergence
        r_feature = torch.norm(module.running_var.data.type(var.type()) - var, 2) + torch.norm(
            module.running_mean.data.type(var.type()) - mean, 2)

        self.r_feature = r_feature
        # must have no output

    def close(self):
        self.hook.remove()


## 最適化対象の準備など

In [10]:
# 損失関数
criterion = nn.CrossEntropyLoss()
kl_loss = nn.KLDivLoss(reduction='batchmean').cuda()


# 最適化対象の入力
bs = 256
size = 56
inputs = torch.randn((bs, 3, size, size),
                     requires_grad=True,
                     device='cuda',
                     dtype=torch.float)

# Optimizerの設定
di_lr = 0.25
optimizer = optim.Adam([inputs], lr=di_lr)

# device = "cuda"
# net = net.to(device).eval()

# var_scale = 2.5e-5
var_scale=0.05
bn_reg_scale = 10
# l2_coeff = 0.0001
l2_coeff = 0

In [ ]:
inputs.data = torch.randn((bs, 3, size, size), requires_grad=True, device='cuda')

optimizer.state = collections.defaultdict(dict)

n_classes = 20
targets = torch.LongTensor([random.randint(0,n_classes) for _ in range(bs)]).to('cuda')
# print("targets: ", targets)

# schedulerの設定
sch = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[50000, 90000], gamma=0.1)

## Create hooks for feature statistics catching
loss_r_feature_layers = []
for module in net.modules():
    if isinstance(module, nn.BatchNorm2d):
        loss_r_feature_layers.append(DeepInversionFeatureHook(module))


# setting up the range for jitter
lim_0, lim_1 = 2, 2

num_iters = 100000
for epoch in range(num_iters):
    # apply random jitter offsets
    off1 = random.randint(-lim_0, lim_0)
    off2 = random.randint(-lim_1, lim_1)
    inputs_jit = torch.roll(inputs, shifts=(off1,off2), dims=(2,3))


    # foward with jit images
    optimizer.zero_grad()
    net.zero_grad()
    outputs = net(inputs_jit)
    logits_all = outputs["logits"]
    logits = logits_all[:, ::4] 
    loss = criterion(logits, targets)
    loss_target = loss.item()

    # apply total variation regularization
    diff1 = inputs_jit[:,:,:,:-1] - inputs_jit[:,:,:,1:]
    diff2 = inputs_jit[:,:,:-1,:] - inputs_jit[:,:,1:,:]
    diff3 = inputs_jit[:,:,1:,:-1] - inputs_jit[:,:,:-1,1:]
    diff4 = inputs_jit[:,:,:-1,:-1] - inputs_jit[:,:,1:,1:]
    loss_var = torch.norm(diff1) + torch.norm(diff2) + torch.norm(diff3) + torch.norm(diff4)
    loss = loss + var_scale*loss_var


    # R_feature loss
    loss_distr = sum([mod.r_feature for mod in loss_r_feature_layers])
    loss = loss + bn_reg_scale*loss_distr # best for noise before BN

    loss = loss + l2_coeff * torch.norm(inputs_jit, 2)


    if (epoch+1) % 10==0:
        print("Epoch: {}, Loss: {}, LossCE: {}, LossVar: {}, LossDistr: {}".format(
            epoch+1, loss.item(), loss_target, loss_var.item(), loss_distr.item()
        ))
    

    if (epoch+1)%5000==0:
        vutils.save_image(inputs.data.clone(),
                        './{}/tiny_size{}_nckass{}_epo{}_lr{}_var{}_bn{}_l2{}.png'.format("debug", size, n_classes, (epoch+1)//200, di_lr, var_scale, bn_reg_scale, l2_coeff),
                        normalize=True, scale_each=True, nrow=10)
    

    loss.backward()
    optimizer.step()
    sch.step()

Epoch: 10, Loss: 812.3821411132812, LossCE: 4.617956161499023, LossVar: 6686.146484375, LossDistr: 47.34568405151367
Epoch: 20, Loss: 659.7679443359375, LossCE: 4.591183185577393, LossVar: 5555.2421875, LossDistr: 37.7414665222168
Epoch: 30, Loss: 624.746826171875, LossCE: 4.5565104484558105, LossVar: 5054.9404296875, LossDistr: 36.74433135986328
Epoch: 40, Loss: 567.0538330078125, LossCE: 4.531551361083984, LossVar: 4792.619140625, LossDistr: 32.28913116455078
Epoch: 50, Loss: 595.61669921875, LossCE: 4.526999473571777, LossVar: 4656.294921875, LossDistr: 35.82749938964844
Epoch: 60, Loss: 553.7542724609375, LossCE: 4.52338981628418, LossVar: 4637.33642578125, LossDistr: 31.736404418945312
Epoch: 70, Loss: 572.96142578125, LossCE: 4.527745246887207, LossVar: 4738.06396484375, LossDistr: 33.153053283691406
Epoch: 80, Loss: 548.78173828125, LossCE: 4.512560844421387, LossVar: 4686.755859375, LossDistr: 30.993141174316406
Epoch: 90, Loss: 569.8635864257812, LossCE: 4.494564533233643, Los

In [66]:
# var_scale=0.05
# bn_reg_scale = 10
# l2_coeff = 0.0001
vutils.save_image(inputs.data.clone(),
                    './{}/tiny_size{}_epo{}_lr{}_var{}_bn{}_l2{}.png'.format("debug", size, (epoch+1)//200, di_lr, var_scale, bn_reg_scale, l2_coeff),
                    normalize=True, scale_each=True, nrow=10)
print(targets)

tensor([66, 25, 99, 81, 85, 21, 32, 72, 50, 94, 17,  4, 12, 67, 11, 67, 46, 27,
        44, 71, 53, 30, 93, 24, 37, 94, 89, 81, 12, 59, 44, 30, 52, 18, 45, 77,
        26, 91, 39, 65, 52, 16, 29, 48, 70, 27, 82,  6, 33,  5, 36, 57,  2, 35,
        48,  4, 55, 88, 79, 37, 91, 75, 21,  7, 41, 28, 93, 38, 24, 68, 50, 74,
        19, 38, 11, 42,  8, 16, 21, 49, 10, 36, 27, 36, 74, 89, 89, 77, 72, 77,
        91, 60, 99, 95, 17, 80, 92, 75,  3, 70, 58, 56, 54, 85, 95,  0, 61, 71,
        38, 94, 71, 60,  1, 80,  0, 51, 57, 34, 88, 84, 15, 55, 34, 23, 50, 36,
        33, 11, 54, 65, 53, 12, 73,  7, 67, 95, 37,  8, 83, 64, 36, 30, 21, 22,
        28, 21, 92, 86, 69,  7,  1,  0, 36, 69, 27, 48, 55,  9, 41, 89, 33, 31,
        13, 93, 11, 94, 99, 35, 38, 47, 79, 92, 48, 80, 73, 99, 24, 96, 17, 31,
        36, 26, 47, 75, 63, 94,  8, 82, 94, 34, 47,  1,  2,  1, 37, 36, 25, 23,
        11, 93, 92, 49, 89, 95, 23, 79, 98, 92, 63,  6, 76, 48, 74, 15,  1, 74,
        64, 30, 30, 61, 57, 94, 12, 41, 